In [ ]:
# Load packages
from pathlib import Path
import os
import re
import sys
import importlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from skyline_qc import *
from skyline_qc.openswath import *

In [ ]:

# Skyline data path
skyline_path = "data/MARTHA/DE17501_Martha_results_subset_07042026.csv"

# Import skyline data (includes column "Isotope Label Type" from Precursor)
skyline_importer = ImportFile(skyline_path)
skyline_data = skyline_importer.import_skyline_file()

In [ ]:
# Isotope Label Type is added in ImportFile.import_skyline_file() (from Precursor).

# qREPs spike levels
qREPs_spike_levels = pd.read_csv('ratio/DE17501_ratio.csv')

# SDRF
sdrf_path = 'sdrf/sdrf_MARTHA_pool.sdrf.tsv'
sdrf_data = pd.read_csv(sdrf_path, sep='\t')



In [ ]:
# OpenMS results from Justin
openms_path = 'openms/openswath_results_export.tsv'
openms_results = pd.read_csv(openms_path, sep='\t')


In [ ]:
# Check file sets match

cross_check_skyline_sdrf(skyline_df = skyline_data, sdrf_df = sdrf_data)

In [ ]:
# Import and use function from output_test.py to get iRT peptides
iRT_peptides = get_irt_peptides(skyline_data)

print('This is the iRT peptides:')
print(iRT_peptides)

In [ ]:
plot_library_dot_product_distribution(skyline_data)

In [ ]:
skyline_clean = filter_library_dot_product(skyline_data, threshold=0.8)


In [ ]:
# Summarise each peptide to count how many heavy or light signals are present in skyline_pivot
peptide_counts = summarise_peptide_counts(skyline_clean)


In [ ]:
report_summary, peptide_list = report_peptide_protein_summary(peptide_counts)

In [ ]:
plot_heavy_light_scatter(peptide_counts)

In [ ]:
filtered_peptide_counts = filter_peptide_counts(peptide_counts, light_cutoff=10, heavy_cutoff=10)

filtered_peptide_counts.head()


In [ ]:
selected_peptides_report,selected_peptides = report_peptide_protein_summary(filtered_peptide_counts)

In [ ]:
from skyline_qc.importer import MergeFiles

skyline_merge = MergeFiles(skyline_data, sdrf_data, selected_peptides).merge_files()


In [ ]:
skyline_merge.head()

In [ ]:
# Batch/Plate adjustment, correcting for each peptide within each plate (batch) 

pool_data = skyline_merge[skyline_merge['characteristics[Sample]'] == 'Pool']
# First, sort dataframe by Plate
pool_data = pool_data.sort_values('characteristics[Plate]')
# Reset index
pool_data = pool_data.reset_index(drop=True)
pool_data.head()

In [ ]:
# Plot log_ratio of each sample in boxplot, colored by plate, but x-axis is Replicate, sorted by plate (though x labels are hidden).
plot_pool_boxplot(pool_data)


In [ ]:
# Calculate intra-plate CV

peptide_plate_stats = calculate_intra_plate_cv(pool_data, col_name='characteristics[Plate]')
plot_intra_plate_cv_stats(peptide_plate_stats, col_name='characteristics[Plate]')

In [ ]:
# Example usage:
plot_inter_plate_cv_kde(peptide_plate_stats)

In [ ]:
# Example usage:
interplate_cv = calculate_inter_plate_cv(peptide_plate_stats)
interplate_cv.head()

In [ ]:
plot_cumulative_peptide_count_by_cv(interplate_cv)

In [ ]:
# Batch/Plate adjustment, correcting for each peptide within each plate (batch) 
# using only peptides in the top 10% lowest inter-plate CV for normalization



top10cv_peptides, pool_selected_df = extract_top_percentile(
    interplate_cv,
    column='inter_plate_cv',
    percentile=0.1,
    id_col='Peptide Sequence',
    source_df=pool_data,
    source_col='Peptide Sequence'
)

# Filter pool_data to include only those peptides for normalization calculation
pool_selected_df

# OpenSWATH



Can you filtering the results by different scores (eg the `VAR_LIBRARY_DOTPROD`, this should be similar / the same as Skylines dotp), and compare the identified RT apex and Intensity with Skylines. You can probably remove the decoys for this. So do

1. filter tsv for `decoy = 0`
2. filter tsv for `VAR_LIBRARY_DOTPROD >= 0.95` (or whatever threshold you use in Skyline)
3. group by `run_id`, and `transition_group_id` to filter for the best peak_group per precursor, run.

In [ ]:
# Import OpenSWATH results

openswath_df = import_openswath_file(openms_path, remove_file_path=True)

In [ ]:
openswath_df

In [ ]:
# I want to look at full column names
openswath_df.columns

In [ ]:
# Plot KDE of VAR_LIBRARY_DOTPROD
# Color by decoy
for decoy_value, color in zip([0, 1], ["b", "r"]):
    subset = openswath_df[openswath_df['decoy'] == decoy_value]
    label = f"decoy = {decoy_value}"
    sns.kdeplot(subset['VAR_LIBRARY_DOTPROD'], shade=True, color=color, label=label)
plt.legend()
# Add horizontal line at 0.95
plt.axvline(x=0.95, color='red', linestyle='--')
plt.show()


In [ ]:
# pelase also plot pep kde 
sns.kdeplot(openswath_df['pep'], shade=True)
# x log scale
plt.xscale('log')
plt.show()


In [ ]:
# filtered_openswath_df = filter_best_peak_group(openswath_df)


In [ ]:
# filtered_openswath_df.shape